In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              mean_absolute_error, r2_score, silhouette_score)
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings('ignore')

## Load & Pre-Transformation Patches

In [2]:
df = pd.read_csv('CGD_Dataset_after_cleaning.csv')
df['TS']        = pd.to_datetime(df['TS'],        errors='coerce')
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce')
df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)

# P1 — Negative pressure → reverse direction, abs magnitude
df['Pressure_Direction_Flag'] = 'Forward'
df.loc[df['Pressure'] < 0, 'Pressure_Direction_Flag'] = 'Reverse'
df['Pressure']         = df['Pressure'].abs()
df['Pressure_Imputed'] = df['Pressure'].isnull()
df['Pressure']         = df.groupby('Meter_ID')['Pressure'].ffill()

# P2 — DMA Zone (60 meters split into 4 zones of 15)
def assign_dma(m):
    n = int(m.split('-')[1])
    return 'DMA-A' if n<=115 else 'DMA-B' if n<=130 else 'DMA-C' if n<=145 else 'DMA-D'
df['DMA_Zone'] = df['Meter_ID'].apply(assign_dma)

# P3 — One fixed GPS per meter (mode broadcast)
df['Latitude']  = df.groupby('Meter_ID')['Latitude'].transform(
    lambda x: x.mode()[0] if x.notna().any() else np.nan)
df['Longitude'] = df.groupby('Meter_ID')['Longitude'].transform(
    lambda x: x.mode()[0] if x.notna().any() else np.nan)

# P4 — Fill residual nulls in calibrated columns
for col in ['Flow_calibrated', 'Energy_calibrated', 'Flow_std']:
    df[col] = df.groupby('Meter_ID')[col].ffill()
    df[col] = df.groupby('Meter_ID')[col].bfill()

# P5 — Cal_Verified flag
df['Cal_Verified'] = (df['Cal_Coefficient'] - 1.0).abs() > 0.002

# P6 — Per-customer-type pressure violation thresholds
PRESSURE_CEILING = {'domestic': 0.5, 'commercial': 4.0, 'industrial': 70.0}
df['Pressure_Violation'] = df.apply(
    lambda r: r['Pressure'] > PRESSURE_CEILING.get(r['Customer_Type'], 70.0), axis=1)

# P7 — Shared time and zone features (required by multiple T-UCs)
df['Month']  = df['TS'].dt.month
df['Hour']   = df['TS'].dt.hour
df['Date']   = df['TS'].dt.date
season_map = {12:'Winter',1:'Winter',2:'Winter',3:'Summer',4:'Summer',5:'Summer',
              6:'Monsoon',7:'Monsoon',8:'Monsoon',9:'PostMonsoon',10:'PostMonsoon',11:'PostMonsoon'}
df['Season'] = df['Month'].map(season_map)
df['Zone']   = pd.cut(df['Latitude'], bins=[0,12.9,13.1,90],
                      labels=['Zone_A','Zone_B','Zone_C'])

# Encoded categoricals (used by ML models)
df['CustomerType_Code'] = df['Customer_Type'].map({'domestic':0,'commercial':1,'industrial':2})
df['Season_Code']       = df['Month'].map({12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
df['Zone_Code']         = df['Zone'].map({'Zone_A':0,'Zone_B':1,'Zone_C':2})

print("Dataset ready:", df.shape)
print("Columns:", df.columns.tolist())


Dataset ready: (6311, 52)
Columns: ['Reading_ID', 'Meter_ID', 'TS', 'Pressure', 'Flow', 'Energy', 'Unit', 'Customer_ID', 'Leak_Flag', 'Maintenance_Flag', 'Notes_Comments', 'Base_Temp_C', 'Base_Pressure_kPa', 'Cal_Coefficient', 'Cal_Version', 'Latitude', 'Longitude', 'Flow_Outlier', 'Pressure_Outlier', 'Customer_Type', 'Join_Date', 'Is_Active', 'NTP_Offset_Sec', 'Asset_Valid', 'Negative_Flow', 'Negative_Energy', 'Direction_Flag', 'Flow_calibrated', 'Energy_calibrated', 'Is_Billable', 'Flow_std', 'Flow_same', 'Pressure_same', 'Energy_same', 'Sensor_Stuck', 'Sensor_Reliable', 'Meter_Capacity_SCMH', 'Capacity_Exceeded', 'Reading_Plausible', 'Pressure_Direction_Flag', 'Pressure_Imputed', 'DMA_Zone', 'Cal_Verified', 'Pressure_Violation', 'Month', 'Hour', 'Date', 'Season', 'Zone', 'CustomerType_Code', 'Season_Code', 'Zone_Code']


---
## T1 — Hourly / Daily Aggregation with Energy Normalization

In [3]:
# Aggregates readings into hourly and daily buckets per meter.
# Uses Flow_std (base-condition adjusted) and Energy_calibrated (calibration corrected)
# — the physically correct columns. Raw Flow/Energy must NOT be used here.

df['Hour_bucket'] = df['TS'].dt.floor('h')

hourly_df = (
    df.groupby(['Meter_ID', 'DMA_Zone', 'Hour_bucket'])
      .agg(
          Flow_std_hourly  = ('Flow_std',          'sum'),
          Energy_hourly    = ('Energy_calibrated', 'sum'),
          Avg_Pressure     = ('Pressure',          'mean'),
          Read_Count       = ('Reading_ID',        'count'),
          Billable_Reads   = ('Is_Billable',       'sum')
      )
      .reset_index()
)

daily_df = (
    df.groupby(['Meter_ID', 'DMA_Zone', 'Date'])
      .agg(
          Flow_std_daily   = ('Flow_std',          'sum'),
          Energy_daily     = ('Energy_calibrated', 'sum'),
          Avg_Pressure     = ('Pressure',          'mean'),
          Read_Count       = ('Reading_ID',        'count'),
          Billable_Reads   = ('Is_Billable',       'sum')
      )
      .reset_index()
)

print("Hourly shape:", hourly_df.shape)
print("Daily  shape:", daily_df.shape)
print("\nHourly sample:")
print(hourly_df.head(5).to_string(index=False))
print("\nDaily sample:")
print(daily_df.head(5).to_string(index=False))


Hourly shape: (6309, 8)
Daily  shape: (5927, 8)

Hourly sample:
Meter_ID DMA_Zone         Hour_bucket  Flow_std_hourly  Energy_hourly  Avg_Pressure  Read_Count  Billable_Reads
 MET-101    DMA-A 2024-01-01 05:00:00         1066.857          4.150          2.53           1               1
 MET-101    DMA-A 2024-01-08 10:00:00         1036.565          3.850          6.55           1               1
 MET-101    DMA-A 2024-01-10 20:00:00          230.373          0.900          3.31           1               1
 MET-101    DMA-A 2024-01-15 13:00:00          470.926          1.870          6.60           1               1
 MET-101    DMA-A 2024-01-18 08:00:00         1064.690          4.303          6.60           1               1

Daily sample:
Meter_ID DMA_Zone       Date  Flow_std_daily  Energy_daily  Avg_Pressure  Read_Count  Billable_Reads
 MET-101    DMA-A 2024-01-01        1066.857         4.150          2.53           1               1
 MET-101    DMA-A 2024-01-08        1036.565   

---
## T2 — Unaccounted-For Gas (UFG) by Zone

In [4]:
# UFG = Gas metered into the network − Gas successfully billed to customers.
# Regulators (PNGRB) cap UFG at a maximum percentage; exceeding it triggers audits.
# Gas_Input  = all Flow_std readings entering the DMA zone
# Gas_Billed = Flow_std from Is_Billable == True rows only (excludes maintenance)

gas_input = (
    df.groupby('DMA_Zone')['Flow_std']
      .sum()
      .reset_index(name='Gas_Input_Sm3')
)
gas_billed = (
    df[df['Is_Billable'] == True]
      .groupby('DMA_Zone')['Flow_std']
      .sum()
      .reset_index(name='Gas_Billed_Sm3')
)

ufg_df = gas_input.merge(gas_billed, on='DMA_Zone')
ufg_df['UFG_Sm3']    = ufg_df['Gas_Input_Sm3'] - ufg_df['Gas_Billed_Sm3']
ufg_df['UFG_Percent'] = (ufg_df['UFG_Sm3'] / ufg_df['Gas_Input_Sm3'] * 100).round(2)
ufg_df['Status']      = ufg_df['UFG_Percent'].apply(
    lambda x: 'CRITICAL' if x > 15 else ('WARNING' if x > 10 else 'OK'))

print("UFG by DMA Zone:")
print(ufg_df.to_string(index=False))


UFG by DMA Zone:
DMA_Zone  Gas_Input_Sm3  Gas_Billed_Sm3      UFG_Sm3  UFG_Percent   Status
   DMA-A   16339217.053    10635346.450  5703870.603        34.91 CRITICAL
   DMA-B   17913395.767    13591918.550  4321477.217        24.12 CRITICAL
   DMA-C   22256507.917    17927101.785  4329406.132        19.45 CRITICAL
   DMA-D   26115709.517    15378152.567 10737556.950        41.12 CRITICAL


---
## T3 — Pressure Violation Counters and Duration per Meter

In [5]:
# Counts pressure exceedances per meter and estimates total violation duration.
# Uses Pressure_Violation flag (per-customer-type thresholds from patch P6):
#   domestic <= 0.5 bar | commercial <= 4.0 bar | industrial <= 70.0 bar
# Regulators require every exceedance to be detected, logged, and resolved.

READING_INTERVAL_MIN = 15   # typical SCADA/AMI interval

violation_df = (
    df.groupby(['Meter_ID', 'Customer_Type', 'DMA_Zone'])
      .agg(
          Total_Readings        = ('Reading_ID',         'count'),
          Violation_Count       = ('Pressure_Violation', 'sum'),
          Max_Pressure          = ('Pressure',           'max'),
          Avg_Pressure          = ('Pressure',           'mean')
      )
      .reset_index()
)
violation_df['Violation_Duration_Min'] = (
    violation_df['Violation_Count'] * READING_INTERVAL_MIN
)
violation_df['Violation_Rate_%'] = (
    violation_df['Violation_Count'] / violation_df['Total_Readings'] * 100
).round(2)

print("Pressure violations (top 10 by count):")
print(violation_df.sort_values('Violation_Count', ascending=False)
      .head(10).to_string(index=False))
print(f"\nTotal violations: {violation_df['Violation_Count'].sum():,}")
print(f"Meters with zero violations: {(violation_df['Violation_Count']==0).sum()}")


Pressure violations (top 10 by count):
Meter_ID Customer_Type DMA_Zone  Total_Readings  Violation_Count  Max_Pressure  Avg_Pressure  Violation_Duration_Min  Violation_Rate_%
 MET-155      domestic    DMA-D             131              130          55.8      5.939008                    1950             99.24
 MET-119      domestic    DMA-B             118              118          63.7      5.619831                    1770            100.00
 MET-138      domestic    DMA-C             117              116          67.8      5.875214                    1740             99.15
 MET-110      domestic    DMA-A             115              115          62.2      6.467217                    1725            100.00
 MET-103      domestic    DMA-A             114              114          52.2      6.100175                    1710            100.00
 MET-108      domestic    DMA-A             110              109          53.5      5.120182                    1635             99.09
 MET-117      do

---
## T4 — Leak Propensity Score + ML Leak Risk Classifier

In [6]:
# STEP 1: Rule-based propensity score (domain formula)
# Score = 0.5×PressureDrop + 0.3×FlowOutlier + 0.2×LeakAlarm
# High_Leak_Risk = score >= 0.5

df['Pressure_Drop']      = df.groupby('Meter_ID')['Pressure'].diff().fillna(0)
df['Flow_Change']        = df.groupby('Meter_ID')['Flow_calibrated'].diff().fillna(0)
df['Pressure_Drop_Flag'] = df['Pressure_Drop'] < -0.3
df['Leak_Alarm']         = df['Leak_Flag'].map({'Yes':1,'No':0,'Unknown':0}).fillna(0)
df['Flow_Anomaly_Int']   = df['Flow_Outlier'].astype(int)

df['Leak_Propensity_Score'] = (
    0.5 * df['Pressure_Drop_Flag'].astype(int) +
    0.3 * df['Flow_Anomaly_Int'] +
    0.2 * df['Leak_Alarm']
).round(3)
df['High_Leak_Risk'] = df['Leak_Propensity_Score'] >= 0.5

print("Score distribution:")
print(df['Leak_Propensity_Score'].value_counts().sort_index().to_string())
print(f"\nHigh risk rows: {df['High_Leak_Risk'].sum()} / {len(df)}")


Score distribution:
Leak_Propensity_Score
0.0    3451
0.2     194
0.3      56
0.5    2422
0.7     154
0.8      31
1.0       3

High risk rows: 2610 / 6311


In [7]:
# STEP 2: Random Forest Leak Risk Classifier
# Target: High_Leak_Risk (derived from propensity score above)
# This lets the model learn the non-linear interaction between signals
# and allows custom-input prediction at the end of this cell.
#
# Manual SMOTE balancing — implemented without external libraries.

def manual_smote(X_min, n_synth, k=5, seed=42):
    rng = np.random.RandomState(seed)
    nn  = NearestNeighbors(n_neighbors=k+1).fit(X_min)
    _, idx = nn.kneighbors(X_min)
    out = []
    for _ in range(n_synth):
        i = rng.randint(0, len(X_min))
        j = idx[i, rng.randint(1, k+1)]
        out.append(X_min[i] + rng.random() * (X_min[j] - X_min[i]))
    return np.array(out)

LEAK_FEATURES = ['Pressure', 'Pressure_Drop', 'Flow_calibrated', 'Flow_Change',
                 'Energy_calibrated', 'Flow_Outlier', 'Pressure_Outlier',
                 'Hour', 'Month', 'Base_Pressure_kPa', 'Base_Temp_C', 'CustomerType_Code']

df_ml = df.dropna(subset=LEAK_FEATURES).copy()
X_leak = df_ml[LEAK_FEATURES].replace([np.inf,-np.inf], 0).fillna(0).values
y_leak = df_ml['High_Leak_Risk'].astype(int).values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42, stratify=y_leak)

# Balance training set with SMOTE
X_min    = X_tr[y_tr == 1]
n_synth  = (y_tr == 0).sum() - (y_tr == 1).sum()
X_synth  = manual_smote(X_min, n_synth)
X_bal    = np.vstack([X_tr, X_synth])
y_bal    = np.concatenate([y_tr, np.ones(n_synth, dtype=int)])

rf_leak = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_leak.fit(X_bal, y_bal)
y_pred = rf_leak.predict(X_te)

print("Leak Risk Classifier Performance:")
print(classification_report(y_te, y_pred, target_names=['Low Risk','High Risk']))
print("Confusion Matrix:")
print(confusion_matrix(y_te, y_pred))
print("\nTop feature importances:")
for f, imp in sorted(zip(LEAK_FEATURES, rf_leak.feature_importances_),
                     key=lambda x: -x[1])[:8]:
    print(f"  {f:<25} {imp:.4f}")


Leak Risk Classifier Performance:
              precision    recall  f1-score   support

    Low Risk       0.99      1.00      1.00       740
   High Risk       1.00      0.99      1.00       522

    accuracy                           1.00      1262
   macro avg       1.00      1.00      1.00      1262
weighted avg       1.00      1.00      1.00      1262

Confusion Matrix:
[[740   0]
 [  4 518]]

Top feature importances:
  Pressure_Drop             0.8374
  Pressure                  0.1265
  Base_Temp_C               0.0046
  Month                     0.0044
  Flow_Change               0.0044
  Flow_calibrated           0.0042
  Energy_calibrated         0.0036
  Base_Pressure_kPa         0.0036


In [8]:
# ── CUSTOM INPUT: Predict Leak Risk for any reading ──────────────────────────
# Fill in the values below and run this cell to get a prediction.

custom_reading = {
    'Pressure'         : 4.5,    # bar
    'Pressure_Drop'    : -0.8,   # bar change from previous read (negative = drop)
    'Flow_calibrated'  : 1200.0, # SCMH
    'Flow_Change'      : -300.0, # SCMH change from previous read
    'Energy_calibrated': 4.5,    # MMBtu
    'Flow_Outlier'     : 0,      # 1 = outlier flagged, 0 = normal
    'Pressure_Outlier' : 0,      # 1 = outlier flagged, 0 = normal
    'Hour'             : 2,      # 0–23
    'Month'            : 6,      # 1–12
    'Base_Pressure_kPa': 101.325,# kPa
    'Base_Temp_C'      : 20.0,   # °C
    'CustomerType_Code': 1,      # 0=domestic, 1=commercial, 2=industrial
}

X_custom = np.array([[custom_reading[f] for f in LEAK_FEATURES]])
pred_class = rf_leak.predict(X_custom)[0]
pred_proba = rf_leak.predict_proba(X_custom)[0]

print("─" * 45)
print("LEAK RISK PREDICTION")
print("─" * 45)
print(f"  Classification : {'HIGH RISK 🔴' if pred_class==1 else 'LOW RISK  🟢'}")
print(f"  Low  Risk prob : {pred_proba[0]*100:.1f}%")
print(f"  High Risk prob : {pred_proba[1]*100:.1f}%")
print("─" * 45)


─────────────────────────────────────────────
LEAK RISK PREDICTION
─────────────────────────────────────────────
  Classification : HIGH RISK 🔴
  Low  Risk prob : 0.6%
  High Risk prob : 99.4%
─────────────────────────────────────────────


---
## T5 — Thermal Energy Conversion (SCMH → MMBtu)

In [9]:
# Natural gas calorific value ≈ 38 MJ/Sm³.
# 1 MMBtu = 1055.056 MJ.
# Energy_MMBtu is the industry billing unit in many markets.

CALORIFIC_VALUE_MJ_PER_SCM = 38.0
MJ_PER_MMBTU               = 1055.056

df['Energy_MMBtu'] = (
    df['Flow_calibrated'] * CALORIFIC_VALUE_MJ_PER_SCM / MJ_PER_MMBTU
).round(4)

print("Energy_MMBtu — stats:")
print(df['Energy_MMBtu'].describe().round(4).to_string())
print("\nSample:")
print(df[['Meter_ID','TS','Flow_calibrated','Energy_MMBtu']].head(8).to_string(index=False))


Energy_MMBtu — stats:
count      6311.0000
mean        480.1301
std        6197.1158
min           1.7918
25%          22.5107
50%          43.3069
75%          69.4336
max      156854.2333

Sample:
Meter_ID                  TS  Flow_calibrated  Energy_MMBtu
 MET-101 2024-01-01 05:00:00         1099.000       39.5827
 MET-101 2024-01-08 10:05:00         1019.000       36.7014
 MET-101 2024-01-10 20:00:30          232.000        8.3560
 MET-101 2024-01-15 13:00:00          477.000       17.1801
 MET-101 2024-01-18 08:00:00         1100.900       39.6512
 MET-101 2024-01-20 02:00:00          300.398       10.8194
 MET-101 2024-02-01 09:00:00         1204.000       43.3645
 MET-101 2024-02-09 02:59:00         1204.000       43.3645


---
## T6 — Customer Consumption Profiles and Seasonal Indices

In [10]:
# Monthly and seasonal consumption per customer.
# Seasonal Index = monthly flow / customer's own yearly average.
# Index > 1.0 → above-average month; < 1.0 → below-average.

customer_monthly = (
    df.groupby(['Customer_ID', 'Customer_Type', 'Month'])
      .agg(
          Monthly_Flow   = ('Flow_calibrated',  'sum'),
          Monthly_Energy = ('Energy_calibrated','sum')
      )
      .reset_index()
)
customer_seasonal = (
    df.groupby(['Customer_ID', 'Customer_Type', 'Season'])
      .agg(
          Seasonal_Flow   = ('Flow_calibrated',  'sum'),
          Seasonal_Energy = ('Energy_calibrated','sum')
      )
      .reset_index()
)

yearly_avg = (
    customer_monthly.groupby('Customer_ID')['Monthly_Flow']
    .mean().reset_index(name='Yearly_Avg_Flow')
)
customer_monthly = customer_monthly.merge(yearly_avg, on='Customer_ID')
customer_monthly['Seasonal_Index'] = (
    customer_monthly['Monthly_Flow'] / customer_monthly['Yearly_Avg_Flow']
).round(4)

print("Monthly profiles:", customer_monthly.shape)
print(customer_monthly.head(8).to_string(index=False))
print("\nSeasonal profiles:", customer_seasonal.shape)
print(customer_seasonal.head(8).to_string(index=False))


Monthly profiles: (612, 7)
Customer_ID Customer_Type  Month  Monthly_Flow  Monthly_Energy  Yearly_Avg_Flow  Seasonal_Index
       C701      domestic      1      4758.085          17.565        43420.425          0.1096
       C701      domestic      2      6487.211          24.494        43420.425          0.1494
       C701      domestic      3     11366.314          42.968        43420.425          0.2618
       C701      domestic      4      3821.620          14.406        43420.425          0.0880
       C701      domestic      5    446200.062          12.468        43420.425         10.2763
       C701      domestic      6      8500.338          31.641        43420.425          0.1958
       C701      domestic      7      5413.686          21.110        43420.425          0.1247
       C701      domestic      8      5377.454          18.674        43420.425          0.1238

Seasonal profiles: (204, 5)
Customer_ID Customer_Type      Season  Seasonal_Flow  Seasonal_Energy
       C70

---
## T7 — Peak Demand Window Detection and Diversity Factor

In [11]:
# Diversity Factor = ΣCustomer_Peaks / System_Coincident_Peak
# A value > 1.0 means customers don't all peak simultaneously — used in
# network capacity planning.

hourly_demand = (
    df.groupby(['Date','Hour'])
      .agg(System_Demand=('Flow_calibrated','sum'))
      .reset_index()
)
peak_row      = hourly_demand.loc[hourly_demand['System_Demand'].idxmax()]
system_peak   = hourly_demand['System_Demand'].max()
customer_peak = (
    df.groupby('Customer_ID')['Flow_calibrated']
      .max().reset_index(name='Customer_Peak_Demand')
)
diversity_factor = customer_peak['Customer_Peak_Demand'].sum() / system_peak

df['Peak_Window'] = (
    (df['TS'].dt.hour == int(peak_row['Hour'])) &
    (df['TS'].dt.date == peak_row['Date'])
)

print(f"Peak Demand Window : {peak_row['Date']}  Hour {int(peak_row['Hour']):02d}:00")
print(f"System Peak Demand : {system_peak:,.2f} SCMH")
print(f"Diversity Factor   : {diversity_factor:.4f}")
print(f"Peak_Window rows   : {df['Peak_Window'].sum()}")
print("\nTop 10 customers by individual peak:")
print(customer_peak.sort_values('Customer_Peak_Demand', ascending=False)
      .head(10).to_string(index=False))


Peak Demand Window : 2025-06-12  Hour 15:00
System Peak Demand : 4,355,000.00 SCMH
Diversity Factor   : 13.0869
Peak_Window rows   : 1

Top 10 customers by individual peak:
Customer_ID  Customer_Peak_Demand
       C854             4355000.0
       C750             4297405.0
       C787             3853170.0
       C843             3483000.0
       C797             3127560.0
       C726             2929575.0
       C771             2788000.0
       C740             2743290.0
       C883             2590980.0
       C851             2580550.0


---
## T8 — Revenue Assurance Metrics

In [12]:
# A successful read requires: sensor reliable + NORMAL operation + flow present.
# Estimated reads are billable but must be flagged on the customer invoice.

df['Read_Success']  = (
    (df['Sensor_Reliable'] == True) &
    (df['Maintenance_Flag'] == 'NORMAL') &
    (df['Flow_calibrated'].notna())
)
df['Estimated_Read'] = ~df['Read_Success']

read_metrics = (
    df.groupby(['Meter_ID','Customer_Type','DMA_Zone'])
      .agg(
          Total_Reads      = ('TS',            'count'),
          Successful_Reads = ('Read_Success',  'sum'),
          Estimated_Reads  = ('Estimated_Read','sum')
      )
      .reset_index()
)
read_metrics['Read_Success_Rate_%'] = (
    read_metrics['Successful_Reads'] / read_metrics['Total_Reads'] * 100
).round(2)

print(f"Network-wide read success rate: {read_metrics['Read_Success_Rate_%'].mean():.2f}%")
print("\nWorst-performing meters:")
print(read_metrics.sort_values('Read_Success_Rate_%')
      .head(10).to_string(index=False))


Network-wide read success rate: 72.37%

Worst-performing meters:
Meter_ID Customer_Type DMA_Zone  Total_Reads  Successful_Reads  Estimated_Reads  Read_Success_Rate_%
 MET-159      domestic    DMA-D           90                57               33                63.33
 MET-146    commercial    DMA-D          102                65               37                63.73
 MET-129    industrial    DMA-B          122                80               42                65.57
 MET-120    commercial    DMA-B          102                67               35                65.69
 MET-141    commercial    DMA-C          114                75               39                65.79
 MET-121    commercial    DMA-B          121                80               41                66.12
 MET-126    commercial    DMA-B          111                74               37                66.67
 MET-109    industrial    DMA-A           96                64               32                66.67
 MET-132    commercial    

---
## T9 — DMA Balance for Loss Localisation

In [13]:
# DMA balance = Gas flowing in − Gas consumed (billable only).
# High loss in a specific DMA tells field crews exactly where to look.
# Uses Flow_std (physically correct) not raw Flow.

dma_input = (
    df.groupby('DMA_Zone')['Flow_std']
      .sum().reset_index(name='Gas_Input_Sm3')
)
dma_consumed = (
    df[df['Is_Billable'] == True]
      .groupby('DMA_Zone')['Flow_std']
      .sum().reset_index(name='Gas_Consumed_Sm3')
)
dma_balance = dma_input.merge(dma_consumed, on='DMA_Zone')
dma_balance['DMA_Loss_Sm3'] = dma_balance['Gas_Input_Sm3'] - dma_balance['Gas_Consumed_Sm3']
dma_balance['Loss_Percent'] = (
    dma_balance['DMA_Loss_Sm3'] / dma_balance['Gas_Input_Sm3'] * 100
).round(2)
dma_balance['Alert'] = dma_balance['Loss_Percent'].apply(
    lambda x: 'CRITICAL' if x>30 else ('WARNING' if x>20 else 'OK'))

print("DMA Balance:")
print(dma_balance.to_string(index=False))
print(f"\nHighest loss zone: {dma_balance.loc[dma_balance['Loss_Percent'].idxmax(),'DMA_Zone']}")


DMA Balance:
DMA_Zone  Gas_Input_Sm3  Gas_Consumed_Sm3  DMA_Loss_Sm3  Loss_Percent    Alert
   DMA-A   16339217.053      10635346.450   5703870.603         34.91 CRITICAL
   DMA-B   17913395.767      13591918.550   4321477.217         24.12  WARNING
   DMA-C   22256507.917      17927101.785   4329406.132         19.45       OK
   DMA-D   26115709.517      15378152.567  10737556.950         41.12 CRITICAL

Highest loss zone: DMA-D


---
## T10 — Anomaly Flags (Night Flow, Reverse Flow)

In [14]:
# Night flow (00:00–05:59) above threshold is suspicious — most meters should
# be near-zero at night. Reverse flow uses Direction_Flag, NOT Flow < 0
# (which is always False after abs() was applied in cleaning UC13 + Patch 1).

NIGHT_FLOW_THRESHOLD = 20   # SCMH

df['Night_Flow_Anomaly'] = (
    (df['Hour'] >= 0) & (df['Hour'] <= 5) &
    (df['Flow_calibrated'] > NIGHT_FLOW_THRESHOLD)
)
df['Reverse_Flow_Flag'] = df['Direction_Flag'] == 'Reverse'
df['Flow_Anomaly_Flag'] = df['Night_Flow_Anomaly'] | df['Reverse_Flow_Flag']

anomaly_summary = (
    df.groupby(['Meter_ID','Customer_Type','DMA_Zone'])
      .agg(
          Night_Flow_Events   = ('Night_Flow_Anomaly','sum'),
          Reverse_Flow_Events = ('Reverse_Flow_Flag', 'sum'),
          Total_Anomalies     = ('Flow_Anomaly_Flag', 'sum')
      )
      .reset_index()
)
print("Anomaly summary (top 10):")
print(anomaly_summary.sort_values('Total_Anomalies', ascending=False)
      .head(10).to_string(index=False))
print(f"\nTotal night-flow events : {df['Night_Flow_Anomaly'].sum()}")
print(f"Total reverse-flow events: {df['Reverse_Flow_Flag'].sum()}")


Anomaly summary (top 10):
Meter_ID Customer_Type DMA_Zone  Night_Flow_Events  Reverse_Flow_Events  Total_Anomalies
 MET-125    commercial    DMA-B                 40                    1               41
 MET-131    commercial    DMA-C                 40                    3               41
 MET-121    commercial    DMA-B                 40                    2               40
 MET-101    industrial    DMA-A                 39                    0               39
 MET-132    commercial    DMA-C                 39                    1               39
 MET-141    commercial    DMA-C                 38                    1               39
 MET-116    commercial    DMA-B                 38                    1               39
 MET-127    industrial    DMA-B                 37                    1               38
 MET-158    commercial    DMA-D                 36                    3               38
 MET-126    commercial    DMA-B                 37                    1             

---
## T11 — Outage Timeline Construction from Event Logs

In [15]:
# Detects OUTAGE_START / OUTAGE_END transitions per meter.
# An outage = any reading where flow is zero, pressure below minimum,
# a leak is confirmed, or the meter is in maintenance/shutdown.
# Leak_Flag and Maintenance_Flag are STRING columns — using ==1 (original bug)
# silently returns all-False. Correct comparisons are used here.

df['outage_flag'] = (
    (df['Flow'] == 0) |
    (df['Pressure'] < 5) |
    (df['Leak_Flag'] == 'Yes') |
    (df['Maintenance_Flag'].isin(['MAINTENANCE','SHUTDOWN']))
).astype(int)

df['prev_flag']   = df.groupby('Meter_ID')['outage_flag'].shift(1).fillna(0)
df['event_type']  = None
df.loc[(df['outage_flag']==1)&(df['prev_flag']==0),'event_type'] = 'OUTAGE_START'
df.loc[(df['outage_flag']==0)&(df['prev_flag']==1),'event_type'] = 'OUTAGE_END'

starts = (df[df['event_type']=='OUTAGE_START'][['Meter_ID','TS']]
          .rename(columns={'TS':'Outage_Start'}).reset_index(drop=True))
ends   = (df[df['event_type']=='OUTAGE_END'][['Meter_ID','TS']]
          .rename(columns={'TS':'Outage_End'}).reset_index(drop=True))
outages = pd.concat([starts, ends], axis=1)
outages = outages.loc[:, ~outages.columns.duplicated()]
outages['Duration_Min'] = (
    (outages['Outage_End'] - outages['Outage_Start'])
    .dt.total_seconds() / 60
).round(1)

def classify_outage(row):
    if row['Leak_Flag'] == 'Yes':                               return 'Leak Shutdown'
    if row['Maintenance_Flag'] in ['MAINTENANCE','SHUTDOWN']:   return 'Scheduled Maintenance'
    if row['Flow'] == 0:                                        return 'Supply Failure'
    if row['Pressure'] < 5:                                     return 'Pressure Drop'
    return 'Unknown'
df['Outage_Type'] = df.apply(classify_outage, axis=1)

print(f"Total outage events: {len(outages)}")
print("\nOutage type distribution:")
print(df['Outage_Type'].value_counts().to_string())
print("\nOutage timeline sample:")
print(outages.head(8).to_string(index=False))


Total outage events: 1380

Outage type distribution:
Outage_Type
Pressure Drop            2316
Unknown                  1987
Scheduled Maintenance    1647
Leak Shutdown             361

Outage timeline sample:
Meter_ID        Outage_Start          Outage_End  Duration_Min
 MET-101 2024-01-01 05:00:00 2024-01-08 10:05:00       10385.0
 MET-101 2024-01-10 20:00:30 2024-01-15 13:00:00        6779.5
 MET-101 2024-01-20 02:00:00 2024-02-09 02:59:00       28859.0
 MET-101 2024-02-27 02:00:30 2024-03-02 21:00:00        6899.5
 MET-101 2024-03-12 16:01:00 2024-06-24 06:00:00      149159.0
 MET-101 2024-07-11 00:00:00 2024-07-12 05:59:30        1799.5
 MET-101 2024-08-27 21:00:00 2024-09-21 09:00:00       35280.0
 MET-101 2024-10-10 00:00:00 2025-01-17 23:02:00      143942.0


---
## T12 — Billing Accuracy Reconciliation (AMI vs Billed)

In [16]:
# AMI_Reading  = Energy_calibrated (what the smart meter recorded)
# Billed_Reading = simulated at 95–100% of AMI (no billing invoice in this dataset)
# In production: replace Billed_Reading with a join to the billing system invoice table.
# Gap > 5% of AMI reading is flagged as a discrepancy requiring investigation.

df['AMI_Reading']    = df['Energy_calibrated']
np.random.seed(42)
df['Billed_Reading'] = (
    df['AMI_Reading'] * np.random.uniform(0.95, 1.00, len(df))
).round(3)
df['Billing_Gap']    = (df['AMI_Reading'] - df['Billed_Reading']).round(3)
df['Billing_Discrepancy'] = (
    df['Billing_Gap'].abs() / df['AMI_Reading'].replace(0, np.nan)
) > 0.05

billing_recon = (
    df.groupby(['Meter_ID','Customer_Type','DMA_Zone'])
      .agg(
          Total_AMI_Reading = ('AMI_Reading',         'sum'),
          Total_Billed      = ('Billed_Reading',      'sum'),
          Total_Gap         = ('Billing_Gap',         'sum'),
          Discrepancy_Count = ('Billing_Discrepancy', 'sum')
      )
      .reset_index()
)
billing_recon['Gap_Percent_%'] = (
    billing_recon['Total_Gap'] / billing_recon['Total_AMI_Reading'] * 100
).round(2)

print("Billing reconciliation (sorted by gap):")
print(billing_recon.sort_values('Gap_Percent_%', ascending=False)
      .head(10).to_string(index=False))
print(f"\nMeters with discrepancies: {(billing_recon['Discrepancy_Count']>0).sum()}")


Billing reconciliation (sorted by gap):
Meter_ID Customer_Type DMA_Zone  Total_AMI_Reading  Total_Billed  Total_Gap  Discrepancy_Count  Gap_Percent_%
 MET-159      domestic    DMA-D            786.348       762.883     23.465                  0           2.98
 MET-109    industrial    DMA-A            296.689       288.124      8.565                  0           2.89
 MET-139    industrial    DMA-C            344.051       334.646      9.405                  0           2.73
 MET-140    commercial    DMA-C            728.295       708.516     19.779                  0           2.72
 MET-101    industrial    DMA-A            297.403       289.362      8.041                  0           2.70
 MET-135    commercial    DMA-C            720.020       700.707     19.313                  0           2.68
 MET-154    commercial    DMA-D            254.610       247.802      6.808                  0           2.67
 MET-152    commercial    DMA-D            370.264       360.381      9.883     

---
## T13 — Safety KPI Mart (Leak Response Time, Closure Time)

In [17]:
# PNGRB requires: respond to confirmed leak within 60 min, close within 240 min.
# Response_Time_Min and Closure_Time_Min are SIMULATED — no dispatch log in dataset.
# In production: join to field operations / ticketing system.
# High_Leak_Risk events (from T4) get faster simulated response.

leak_events = df[df['Leak_Flag'] == 'Yes'][[
    'Reading_ID','Meter_ID','Customer_ID','Zone','DMA_Zone',
    'TS','Pressure','High_Leak_Risk','Leak_Propensity_Score'
]].copy().reset_index(drop=True)
leak_events.rename(columns={'TS':'Leak_Detected_At'}, inplace=True)

np.random.seed(42)
n = len(leak_events)
leak_events['Response_Time_Min'] = np.where(
    leak_events['High_Leak_Risk'],
    np.random.randint(15,  60, n),
    np.random.randint(30, 240, n)
)
leak_events['Closure_Time_Min'] = np.where(
    leak_events['High_Leak_Risk'],
    np.random.randint( 60, 240, n),
    np.random.randint(120, 480, n)
)
leak_events['Dispatched_At'] = (leak_events['Leak_Detected_At'] +
    pd.to_timedelta(leak_events['Response_Time_Min'], unit='m'))
leak_events['Closed_At'] = (leak_events['Dispatched_At'] +
    pd.to_timedelta(leak_events['Closure_Time_Min'], unit='m'))
leak_events['Response_Within_SLA'] = leak_events['Response_Time_Min'] <= 60
leak_events['Closure_Within_SLA']  = leak_events['Closure_Time_Min']  <= 240

safety_kpi = (
    leak_events.groupby('DMA_Zone')
      .agg(
          Total_Leaks           = ('Reading_ID',           'count'),
          High_Risk_Leaks       = ('High_Leak_Risk',       'sum'),
          Avg_Response_Min      = ('Response_Time_Min',    'mean'),
          Avg_Closure_Min       = ('Closure_Time_Min',     'mean'),
          Response_SLA_Met      = ('Response_Within_SLA',  'sum'),
          Closure_SLA_Met       = ('Closure_Within_SLA',   'sum')
      )
      .reset_index()
)
safety_kpi['Response_SLA_%'] = (safety_kpi['Response_SLA_Met']/safety_kpi['Total_Leaks']*100).round(2)
safety_kpi['Closure_SLA_%']  = (safety_kpi['Closure_SLA_Met'] /safety_kpi['Total_Leaks']*100).round(2)

print("Safety KPI Mart by DMA Zone:")
print(safety_kpi.round(2).to_string(index=False))


Safety KPI Mart by DMA Zone:
DMA_Zone  Total_Leaks  High_Risk_Leaks  Avg_Response_Min  Avg_Closure_Min  Response_SLA_Met  Closure_SLA_Met  Response_SLA_%  Closure_SLA_%
   DMA-A           91               39             84.76           228.15                48               57           52.75          62.64
   DMA-B          102               46             92.45           234.83                54               61           52.94          59.80
   DMA-C           89               41             93.47           220.42                48               60           53.93          67.42
   DMA-D           79               41             84.22           216.56                46               55           58.23          69.62


---
## T14 — GIS Rollups by Zone

In [18]:
# Spatial aggregation for network planning and PNGRB regulatory reporting.
# Three levels: zone-level KPI rollup, meter-level GPS summary,
# and customer-type flow distribution per zone.

gis_zone = (
    df.groupby('Zone')
      .agg(
          Total_Flow_SCMH     = ('Flow_calibrated',   'sum'),
          Avg_Pressure_bar    = ('Pressure',           'mean'),
          Total_Energy_MMBtu  = ('Energy_MMBtu',       'sum'),
          Meter_Count         = ('Meter_ID',           'nunique'),
          Customer_Count      = ('Customer_ID',        'nunique'),
          Leak_Events         = ('Leak_Flag',          lambda x:(x=='Yes').sum()),
          Pressure_Violations = ('Pressure_Violation', 'sum'),
          Total_Readings      = ('Reading_ID',         'count')
      )
      .reset_index().round(2)
)
print("Zone Rollup:")
print(gis_zone.to_string(index=False))

gis_meter = (
    df.dropna(subset=['Latitude','Longitude'])
      .groupby(['Meter_ID','Zone','Customer_Type'])
      .agg(
          Latitude      = ('Latitude',        'first'),
          Longitude     = ('Longitude',       'first'),
          Total_Flow    = ('Flow_calibrated', 'sum'),
          Avg_Pressure  = ('Pressure',        'mean'),
          Leak_Count    = ('Leak_Flag',       lambda x:(x=='Yes').sum()),
          Readings      = ('Reading_ID',      'count')
      )
      .reset_index().round(4)
)
print("\nMeter Rollup (first 10):")
print(gis_meter.head(10).to_string(index=False))

zone_mix = (
    df.groupby(['Zone','Customer_Type'])['Flow_calibrated']
      .sum().reset_index(name='Total_Flow_SCMH').round(2)
)
print("\nFlow by Zone × Customer Type:")
print(zone_mix.to_string(index=False))


Zone Rollup:
  Zone  Total_Flow_SCMH  Avg_Pressure_bar  Total_Energy_MMBtu  Meter_Count  Customer_Count  Leak_Events  Pressure_Violations  Total_Readings
Zone_A      15348423.59              5.43           552804.87            9               9           58                  661             921
Zone_B      36373464.17              5.75          1310064.72           33              31          203                 2060            3501
Zone_C      32407748.56              5.68          1167231.36           18              18          100                 1253            1889

Meter Rollup (first 10):
Meter_ID   Zone Customer_Type  Latitude  Longitude  Total_Flow  Avg_Pressure  Leak_Count  Readings
 MET-101 Zone_B    industrial   12.9998    80.2055  535498.099        5.4558           7       101
 MET-102 Zone_C      domestic   13.2303    80.1585  161573.355        6.4916           7       106
 MET-103 Zone_C      domestic   13.1428    80.3815  188362.457        6.1002           4       114
 

---
## T15 — Forecast Features + Random Forest Flow Prediction Model

In [19]:
# Random Forest Regressor for gas flow prediction.
# Features: physical readings (Pressure, Energy), time (Hour, Month),
#           customer context (type, season, zone), calibration state.
# R² ≈ 0.83 on test set — meaningful for short-term demand forecasting.

FLOW_FEATURES = ['Pressure','Hour','Month','CustomerType_Code','Season_Code',
                 'Zone_Code','Energy_calibrated','Cal_Coefficient',
                 'Base_Pressure_kPa','Base_Temp_C']
TARGET        = 'Flow_calibrated'

# Remove extreme outliers from training (keep in df)
flow_cap  = df[TARGET].quantile(0.99)
mask      = (df[TARGET] <= flow_cap) & df[FLOW_FEATURES].notna().all(axis=1)
model_df  = df[mask]

X_f = model_df[FLOW_FEATURES].values
y_f = model_df[TARGET].values

X_tr, X_te, y_tr, y_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

rf_flow = RandomForestRegressor(
    n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf_flow.fit(X_tr, y_tr)
y_pred = rf_flow.predict(X_te)

print(f"Flow Prediction Model Performance:")
print(f"  MAE : {mean_absolute_error(y_te, y_pred):,.2f} SCMH")
print(f"  R²  : {r2_score(y_te, y_pred):.4f}")
print("\nFeature importances:")
for f, imp in sorted(zip(FLOW_FEATURES, rf_flow.feature_importances_),
                     key=lambda x:-x[1]):
    print(f"  {f:<25} {imp:.4f}")

# Predict on full df
valid_idx = df[df[FLOW_FEATURES].notna().all(axis=1)].index
df['Flow_Predicted'] = np.nan
df.loc[valid_idx,'Flow_Predicted'] = rf_flow.predict(
    df.loc[valid_idx, FLOW_FEATURES].values).round(2)
print(f"\nFlow_Predicted added — non-null: {df['Flow_Predicted'].notna().sum()}")


Flow Prediction Model Performance:
  MAE : 191.17 SCMH
  R²  : 0.8281

Feature importances:
  Energy_calibrated         0.9396
  Pressure                  0.0148
  Base_Temp_C               0.0131
  Hour                      0.0091
  Month                     0.0057
  Cal_Coefficient           0.0052
  Base_Pressure_kPa         0.0040
  CustomerType_Code         0.0032
  Zone_Code                 0.0028
  Season_Code               0.0025

Flow_Predicted added — non-null: 6310


In [20]:
# ── CUSTOM INPUT: Predict flow for any meter reading ─────────────────────────

custom_flow_input = {
    'Pressure'         : 4.5,    # bar
    'Hour'             : 14,     # 0–23
    'Month'            : 7,      # 1–12
    'CustomerType_Code': 1,      # 0=domestic, 1=commercial, 2=industrial
    'Season_Code'      : 2,      # 0=Winter, 1=Summer, 2=Monsoon, 3=PostMonsoon
    'Zone_Code'        : 1,      # 0=Zone_A, 1=Zone_B, 2=Zone_C
    'Energy_calibrated': 4.2,    # MMBtu
    'Cal_Coefficient'  : 1.002,  # calibration factor (1.0 = no correction)
    'Base_Pressure_kPa': 101.325,# kPa
    'Base_Temp_C'      : 22.0,   # °C
}

X_cust = np.array([[custom_flow_input[f] for f in FLOW_FEATURES]])
pred_flow = rf_flow.predict(X_cust)[0]
# Get prediction interval via individual tree predictions
tree_preds = np.array([t.predict(X_cust)[0] for t in rf_flow.estimators_])

print("─" * 45)
print("FLOW PREDICTION")
print("─" * 45)
print(f"  Predicted Flow : {pred_flow:,.2f} SCMH")
print(f"  90% range      : {np.percentile(tree_preds,5):,.2f} – {np.percentile(tree_preds,95):,.2f} SCMH")
print(f"  Std dev        : ±{tree_preds.std():,.2f} SCMH")
print("─" * 45)


─────────────────────────────────────────────
FLOW PREDICTION
─────────────────────────────────────────────
  Predicted Flow : 1,089.25 SCMH
  90% range      : 983.31 – 1,164.76 SCMH
  Std dev        : ±169.41 SCMH
─────────────────────────────────────────────


---
## T16 — Customer Segmentation (KMeans Clustering)

In [21]:
# KMeans clustering on per-customer usage statistics.
# 3 clusters → Low / Mid / High Consumption segments.
# Drives tariff analysis, demand forecasting, and targeted maintenance.

CLUSTER_FEATURES = ['Avg_Flow','Total_Flow','Avg_Pressure','Avg_Energy',
                    'Leak_Count','Outlier_Count','Months_Active']

cust_stats = df.groupby(['Customer_ID','Customer_Type']).agg(
    Avg_Flow      = ('Flow_calibrated',  'mean'),
    Total_Flow    = ('Flow_calibrated',  'sum'),
    Avg_Pressure  = ('Pressure',         'mean'),
    Avg_Energy    = ('Energy_calibrated','mean'),
    Leak_Count    = ('Leak_Flag',        lambda x: (x=='Yes').sum()),
    Outlier_Count = ('Flow_Outlier',     'sum'),
    Months_Active = ('Month',            'nunique')
).reset_index()

cust_stats_clean = cust_stats.dropna(subset=CLUSTER_FEATURES).copy()

seg_scaler = StandardScaler()
X_seg = seg_scaler.fit_transform(cust_stats_clean[CLUSTER_FEATURES].fillna(0))

km = KMeans(n_clusters=3, random_state=42, n_init=10)
cust_stats_clean['Cluster'] = km.fit_predict(X_seg)

# Name clusters by average flow rank
avg_by_cluster = cust_stats_clean.groupby('Cluster')['Avg_Flow'].mean()
rank_map = avg_by_cluster.rank().astype(int).map(
    {1:'Low_Consumption', 2:'Mid_Consumption', 3:'High_Consumption'})
cust_stats_clean['Segment'] = cust_stats_clean['Cluster'].map(rank_map)

sil = silhouette_score(X_seg, cust_stats_clean['Cluster'])
print(f"Silhouette Score: {sil:.4f}  (range -1 to 1, higher = better)")
print("\nSegment profiles:")
print(cust_stats_clean.groupby('Segment')[CLUSTER_FEATURES].mean().round(2).to_string())
print("\nCustomer type mix per segment:")
print(pd.crosstab(cust_stats_clean['Segment'], cust_stats_clean['Customer_Type']).to_string())

# Merge segment back to main df
df = df.merge(
    cust_stats_clean[['Customer_ID','Segment']],
    on='Customer_ID', how='left')
print(f"\nSegment column added to df. Distribution:")
print(df['Segment'].value_counts().to_string())


Silhouette Score: 0.2250  (range -1 to 1, higher = better)

Segment profiles:
                  Avg_Flow  Total_Flow  Avg_Pressure  Avg_Energy  Leak_Count  Outlier_Count  Months_Active
Segment                                                                                                   
High_Consumption  30238.49  4340797.12          5.44        7.17        9.14           2.43           12.0
Low_Consumption    3208.92   350478.43          5.46        4.54        6.38           1.08           12.0
Mid_Consumption   10714.52  1149768.80          6.36        4.07        6.15           3.08           12.0

Customer type mix per segment:
Customer_Type     commercial  domestic  industrial
Segment                                           
High_Consumption           9         4           1
Low_Consumption           11         7           6
Mid_Consumption            6         3           4

Segment column added to df. Distribution:
Segment
Low_Consumption     2705
High_Consumption    2076

In [22]:
# ── CUSTOM INPUT: Segment any customer by usage profile ──────────────────────

custom_customer = {
    'Avg_Flow'     : 800.0,   # average SCMH
    'Total_Flow'   : 96000.0, # total SCMH over dataset period
    'Avg_Pressure' : 3.8,     # bar
    'Avg_Energy'   : 3.0,     # average MMBtu per read
    'Leak_Count'   : 2,       # number of confirmed leak events
    'Outlier_Count': 1,       # number of outlier-flagged reads
    'Months_Active': 10,      # distinct months with readings (1–12)
}

X_cust_seg = seg_scaler.transform(
    [[custom_customer[f] for f in CLUSTER_FEATURES]])
cluster_id   = km.predict(X_cust_seg)[0]
segment_name = rank_map[cluster_id]

# Distance to each cluster centroid (lower = closer match)
distances = np.linalg.norm(X_cust_seg - km.cluster_centers_, axis=1)

print("─" * 45)
print("CUSTOMER SEGMENT PREDICTION")
print("─" * 45)
print(f"  Segment        : {segment_name}")
print(f"  Cluster ID     : {cluster_id}")
print(f"  Confidence     : distances to centroids:")
for i, (d, name) in enumerate(zip(distances, rank_map.to_dict().values())):
    marker = " ◀ assigned" if i == cluster_id else ""
    print(f"    {name:<20} {d:.3f}{marker}")
print("─" * 45)


─────────────────────────────────────────────
CUSTOMER SEGMENT PREDICTION
─────────────────────────────────────────────
  Segment        : Low_Consumption
  Cluster ID     : 2
  Confidence     : distances to centroids:
    Mid_Consumption      4.570
    High_Consumption     5.156
    Low_Consumption      3.323 ◀ assigned
─────────────────────────────────────────────


---
## T17 — Meter Health Score

In [23]:
# Composite health score per meter combining six signals:
#   - Sensor stuck episodes          (weight 30)
#   - Flow outlier rate              (weight 20)
#   - Pressure outlier rate          (weight 15)
#   - Calibration drift from 1.0     (weight 500 × drift)
#   - Capacity exceeded rate         (weight 25)
#   - Leak event rate                (weight 10)
# Score = 100 − penalties, clipped to [0, 100].
# Score > 70 = Healthy | 40–70 = Maintenance Due | < 40 = Critical

meter_health = df.groupby(['Meter_ID','Customer_Type','DMA_Zone']).agg(
    Reading_Count          = ('Reading_ID',        'count'),
    Stuck_Pct              = ('Sensor_Stuck',       'mean'),
    Outlier_Flow_Pct       = ('Flow_Outlier',       'mean'),
    Outlier_Press_Pct      = ('Pressure_Outlier',   'mean'),
    Cal_Drift              = ('Cal_Coefficient',    lambda x: (x-1.0).abs().mean()),
    Capacity_Exceeded_Pct  = ('Capacity_Exceeded',  'mean'),
    Leak_Rate              = ('Leak_Flag',          lambda x: (x=='Yes').sum()/max(len(x),1))
).reset_index()

meter_health['Health_Score'] = (
    100
    - meter_health['Stuck_Pct']             * 30
    - meter_health['Outlier_Flow_Pct']      * 20
    - meter_health['Outlier_Press_Pct']     * 15
    - meter_health['Cal_Drift']             * 500
    - meter_health['Capacity_Exceeded_Pct'] * 25
    - meter_health['Leak_Rate']             * 100 * 10
).clip(0, 100).round(2)

meter_health['Health_Status'] = pd.cut(
    meter_health['Health_Score'],
    bins=[-1, 40, 70, 101],
    labels=['Critical', 'Maintenance Due', 'Healthy']
)

print("Meter Health Score distribution:")
print(meter_health['Health_Status'].value_counts().to_string())
print("\nHealth score stats:")
print(meter_health['Health_Score'].describe().round(2).to_string())
print("\nCritical meters:")
print(meter_health[meter_health['Health_Status']=='Critical']
      [['Meter_ID','Customer_Type','DMA_Zone','Health_Score','Health_Status']]
      .sort_values('Health_Score').to_string(index=False))


Meter Health Score distribution:
Health_Status
Critical           52
Maintenance Due     8
Healthy             0

Health score stats:
count    60.00
mean     19.54
std      16.28
min       0.00
25%       3.24
50%      17.82
75%      31.33
max      54.34

Critical meters:
Meter_ID Customer_Type DMA_Zone  Health_Score Health_Status
 MET-107    commercial    DMA-A          0.00      Critical
 MET-113    commercial    DMA-A          0.00      Critical
 MET-129    industrial    DMA-B          0.00      Critical
 MET-133    commercial    DMA-C          0.00      Critical
 MET-126    commercial    DMA-B          0.00      Critical
 MET-124      domestic    DMA-B          0.00      Critical
 MET-120    commercial    DMA-B          0.00      Critical
 MET-119      domestic    DMA-B          0.00      Critical
 MET-149    commercial    DMA-D          0.00      Critical
 MET-148      domestic    DMA-D          0.00      Critical
 MET-143      domestic    DMA-C          0.00      Critical
 MET-160

---
## T18 — Regulatory Reporting Tables (PNGRB KPIs)

In [24]:
# PNGRB (Petroleum and Natural Gas Regulatory Board) requires:
#   1. Pressure violation summary (count, duration, resolution rate)
#   2. Leak response and closure times by zone
#   3. UFG by network zone
#   4. Network composition (pipe material simulated — not in raw data)
#   5. Billing read quality metrics

# Pipe material simulated from meter ID range (in production: from GIS/asset register)
pipe_material_map = {
    'DMA-A': 'MDPE',  'DMA-B': 'Steel',
    'DMA-C': 'MDPE',  'DMA-D': 'Steel'
}
df['Pipe_Material'] = df['DMA_Zone'].map(pipe_material_map)

# Table 1 — Pressure violations
reg_pressure = (
    df.groupby(['DMA_Zone','Pipe_Material'])
      .agg(
          Total_Readings        = ('Reading_ID',         'count'),
          Pressure_Violations   = ('Pressure_Violation', 'sum'),
          Violation_Duration_Hr = ('Pressure_Violation', lambda x: x.sum()*15/60)
      )
      .reset_index()
)
reg_pressure['Violation_Rate_%'] = (
    reg_pressure['Pressure_Violations']/reg_pressure['Total_Readings']*100
).round(2)

# Table 2 — UFG per zone
reg_ufg = ufg_df.copy()   # from T2

# Table 3 — Read quality
reg_read = (
    df.groupby('DMA_Zone')
      .agg(
          Total_Reads     = ('Reading_ID',     'count'),
          Successful_Reads= ('Read_Success',   'sum'),
          Estimated_Reads = ('Estimated_Read', 'sum')
      )
      .reset_index()
)
reg_read['Read_Quality_%'] = (reg_read['Successful_Reads']/reg_read['Total_Reads']*100).round(2)

print("REGULATORY TABLE 1 — Pressure Violations:")
print(reg_pressure.to_string(index=False))
print("\nREGULATORY TABLE 2 — UFG by Zone:")
print(reg_ufg.to_string(index=False))
print("\nREGULATORY TABLE 3 — Read Quality:")
print(reg_read.to_string(index=False))


REGULATORY TABLE 1 — Pressure Violations:
DMA_Zone Pipe_Material  Total_Readings  Pressure_Violations  Violation_Duration_Hr  Violation_Rate_%
   DMA-A          MDPE            1543                  888                 222.00             57.55
   DMA-B         Steel            1623                  975                 243.75             60.07
   DMA-C          MDPE            1632                  988                 247.00             60.54
   DMA-D         Steel            1513                 1123                 280.75             74.22

REGULATORY TABLE 2 — UFG by Zone:
DMA_Zone  Gas_Input_Sm3  Gas_Billed_Sm3      UFG_Sm3  UFG_Percent   Status
   DMA-A   16339217.053    10635346.450  5703870.603        34.91 CRITICAL
   DMA-B   17913395.767    13591918.550  4321477.217        24.12 CRITICAL
   DMA-C   22256507.917    17927101.785  4329406.132        19.45 CRITICAL
   DMA-D   26115709.517    15378152.567 10737556.950        41.12 CRITICAL

REGULATORY TABLE 3 — Read Quality:
DMA_Zon

---
## T19 — SLA Breach Flags for Read and Repair

In [25]:
# Two SLA types:
#   Read SLA  : consecutive reads on the same meter should arrive within READ_SLA_HRS
#   Repair SLA: maintenance/shutdown windows should not exceed REPAIR_SLA_HRS
# Breach flags are added to the main df and summarised per meter.

READ_SLA_HRS   = 2.0    # hours — max acceptable gap between consecutive reads
REPAIR_SLA_HRS = 4.0    # hours — max acceptable maintenance window

df['Time_Since_Last_Read_Hr'] = (
    df.groupby('Meter_ID')['TS'].diff().dt.total_seconds() / 3600
)

# Read SLA breach — any gap above threshold
df['Read_SLA_Breach'] = df['Time_Since_Last_Read_Hr'] > READ_SLA_HRS

# Repair SLA breach — only on maintenance/shutdown rows
df['Repair_SLA_Breach'] = (
    df['Maintenance_Flag'].isin(['MAINTENANCE','SHUTDOWN']) &
    (df['Time_Since_Last_Read_Hr'] > REPAIR_SLA_HRS)
)

sla_summary = (
    df.groupby(['Meter_ID','Customer_Type','DMA_Zone'])
      .agg(
          Total_Reads       = ('Reading_ID',        'count'),
          Read_SLA_Breaches = ('Read_SLA_Breach',   'sum'),
          Repair_SLA_Breaches=('Repair_SLA_Breach', 'sum'),
          Avg_Read_Gap_Hr   = ('Time_Since_Last_Read_Hr','mean')
      )
      .reset_index()
)
sla_summary['Read_SLA_Breach_%'] = (
    sla_summary['Read_SLA_Breaches']/sla_summary['Total_Reads']*100
).round(2)

print("SLA breach summary (top 10 by read SLA breach count):")
print(sla_summary.sort_values('Read_SLA_Breaches', ascending=False)
      .head(10).to_string(index=False))
print(f"\nTotal read SLA breaches  : {df['Read_SLA_Breach'].sum():,}")
print(f"Total repair SLA breaches: {df['Repair_SLA_Breach'].sum():,}")


SLA breach summary (top 10 by read SLA breach count):
Meter_ID Customer_Type DMA_Zone  Total_Reads  Read_SLA_Breaches  Repair_SLA_Breaches  Avg_Read_Gap_Hr  Read_SLA_Breach_%
 MET-155      domestic    DMA-D          131                127                   37       179.730769              96.95
 MET-149    commercial    DMA-D          125                123                   39       153.549194              98.40
 MET-131    commercial    DMA-C          122                120                   35       158.364738              98.36
 MET-121    commercial    DMA-B          121                119                   38       157.791667              98.35
 MET-129    industrial    DMA-B          122                118                   39       154.992769              96.72
 MET-135    commercial    DMA-C          120                118                   31       160.437675              98.33
 MET-132    commercial    DMA-C          118                117                   39       163.5390

---
## T20 — Carbon Intensity Estimation from Gas Energy

In [26]:
# Natural gas emission factor: 53.07 kg CO₂ per MMBtu (US EPA default).
# CO₂_Tonnes_per_GJ = 0.0561 (IPCC AR6 default for natural gas).
# Energy_MMBtu from T5 is the input. Results are per-reading and rolled up
# per customer, zone, and month for carbon reporting.

CO2_KG_PER_MMBTU = 53.07    # kg CO₂ per MMBtu (EPA natural gas factor)
CO2_KG_PER_TN    = 1000.0   # kg per tonne

df['CO2_kg']     = (df['Energy_MMBtu'] * CO2_KG_PER_MMBTU).round(3)
df['CO2_Tonnes'] = (df['CO2_kg'] / CO2_KG_PER_TN).round(6)

# Zone-level carbon intensity
carbon_zone = (
    df.groupby(['DMA_Zone','Zone'])
      .agg(
          Total_Energy_MMBtu = ('Energy_MMBtu',    'sum'),
          Total_CO2_Tonnes   = ('CO2_Tonnes',      'sum'),
          Avg_CO2_per_read   = ('CO2_kg',          'mean'),
          Total_Flow_std     = ('Flow_std',        'sum')   # ← add Flow_std here
      )
      .reset_index().round(3)
)
carbon_zone['CO2_intensity_kgPerSm3'] = (
    carbon_zone['Total_CO2_Tonnes'] * 1000 /
    carbon_zone['Total_Flow_std']          # ← use the grouped column, not .values
).round(4)
carbon_zone = carbon_zone.drop(columns='Total_Flow_std')  # drop helper col if not needed

# Monthly carbon trend
carbon_monthly = (
    df.groupby(['Month','Season','Customer_Type'])
      .agg(
          Total_CO2_Tonnes = ('CO2_Tonnes','sum'),
          Avg_Flow         = ('Flow_calibrated','mean')
      )
      .reset_index().round(3)
)

# Customer-level carbon footprint
carbon_customer = (
    df.groupby(['Customer_ID','Customer_Type','Segment'])
      .agg(
          Total_CO2_Tonnes  = ('CO2_Tonnes', 'sum'),
          Total_Energy_MMBtu= ('Energy_MMBtu','sum')
      )
      .reset_index().round(3)
)

print("Carbon by Zone:")
print(carbon_zone.to_string(index=False))
print("\nMonthly Carbon Trend:")
print(carbon_monthly.head(12).to_string(index=False))
print("\nTop 10 highest-emission customers:")
print(carbon_customer.sort_values('Total_CO2_Tonnes', ascending=False)
      .head(10).to_string(index=False))

Carbon by Zone:
DMA_Zone   Zone  Total_Energy_MMBtu  Total_CO2_Tonnes  Avg_CO2_per_read  CO2_intensity_kgPerSm3
   DMA-A Zone_A           95728.535          5080.313         26879.965                  1.9424
   DMA-A Zone_B          256468.252         13610.770         19583.842                  1.9482
   DMA-A Zone_C          248623.851         13194.468         20021.954                  1.9584
   DMA-B Zone_A           25484.430          1352.459         14236.408                  1.9654
   DMA-B Zone_B          469491.422         24915.910         17583.564                  1.9430
   DMA-B Zone_C          165139.719          8763.965         78954.639                  1.9909
   DMA-C Zone_A          341185.714         18106.726         40237.169                  1.9270
   DMA-C Zone_B          172724.164          9166.471         14480.997                  1.9578
   DMA-C Zone_C          298536.459         15843.330         28858.524                  1.9373
   DMA-D Zone_A         

---
## Save Final Output

In [27]:
df.to_csv('CGD_Transformation_Final.csv', index=False)
print("Saved: CGD_Transformation_Final.csv")
print("Final shape:", df.shape)
print("\nAll columns:")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2}. {c}")


Saved: CGD_Transformation_Final.csv
Final shape: (6311, 83)

All columns:
   1. Reading_ID
   2. Meter_ID
   3. TS
   4. Pressure
   5. Flow
   6. Energy
   7. Unit
   8. Customer_ID
   9. Leak_Flag
  10. Maintenance_Flag
  11. Notes_Comments
  12. Base_Temp_C
  13. Base_Pressure_kPa
  14. Cal_Coefficient
  15. Cal_Version
  16. Latitude
  17. Longitude
  18. Flow_Outlier
  19. Pressure_Outlier
  20. Customer_Type
  21. Join_Date
  22. Is_Active
  23. NTP_Offset_Sec
  24. Asset_Valid
  25. Negative_Flow
  26. Negative_Energy
  27. Direction_Flag
  28. Flow_calibrated
  29. Energy_calibrated
  30. Is_Billable
  31. Flow_std
  32. Flow_same
  33. Pressure_same
  34. Energy_same
  35. Sensor_Stuck
  36. Sensor_Reliable
  37. Meter_Capacity_SCMH
  38. Capacity_Exceeded
  39. Reading_Plausible
  40. Pressure_Direction_Flag
  41. Pressure_Imputed
  42. DMA_Zone
  43. Cal_Verified
  44. Pressure_Violation
  45. Month
  46. Hour
  47. Date
  48. Season
  49. Zone
  50. CustomerType_Code
  51. 

In [28]:
len(df.columns)

83